TurkQuish feature-group ablation plots.

Purpose:
Generate manuscript-ready plots showing the contribution of:
- Lexical / structural features
- Adversarial / brand features
- Turkish linguistic features
- Graph-based infrastructure features

Input:
Preferred:
    ablation_outputs/ablation_summary.csv

Alternative:
    step5_incremental.csv from the existing notebook

Outputs:
    ablation_plots/
        fig_ablation_macro_f1.png / .pdf
        fig_ablation_delta_macro_f1.png / .pdf
        fig_ablation_metrics_comparison.png / .pdf
        fig_ablation_efficiency_tradeoff.png / .pdf
        fig_ablation_feature_group_matrix.png / .pdf
        fig_ablation_contribution_summary.png / .pdf

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. CONFIGURATION
# ============================================================

# For Colab, update this to your Drive folder if needed.
DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"

# If running locally, use:
# DATA_DIR = "."

ABLATION_DIR = os.path.join(DATA_DIR, "ablation_outputs")
OUT_DIR = os.path.join(DATA_DIR, "ablation_plots")
os.makedirs(OUT_DIR, exist_ok=True)

PRIMARY_SUMMARY = os.path.join(ABLATION_DIR, "ablation_summary.csv")
PRIMARY_TABLE = os.path.join(ABLATION_DIR, "ablation_manuscript_table.csv")

# Existing notebook fallback
NOTEBOOK_STEP5 = os.path.join(DATA_DIR, "step5_incremental.csv")


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def save_fig(name):
    png_path = os.path.join(OUT_DIR, f"{name}.png")
    pdf_path = os.path.join(OUT_DIR, f"{name}.pdf")

    plt.tight_layout()
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.close()

    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")


def clean_setting_label(text):
    """
    Shortens long ablation-setting names for plotting.
    """
    text = str(text)

    replacements = {
        "Lexical + structural only": "Lexical",
        "Lexical + structural + adversarial/brand": "Lexical + Brand",
        "Lexical + structural + Turkish linguistic": "Lexical + Turkish",
        "Lexical + structural + graph-based infrastructure": "Lexical + Graph",
        "Lexical + structural + adversarial/brand + Turkish linguistic": "Lexical + Brand + Turkish",
        "Lexical + structural + adversarial/brand + Turkish linguistic + graph": "Lexical + Brand + Turkish + Graph",
        "Full retained feature set, artifact-controlled": "Full retained",
        "Full retained feature set, artifact-controlled, inductive graph projection": "Full retained + Inductive graph",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = text.replace("A1: ", "A1\n")
    text = text.replace("A2: ", "A2\n")
    text = text.replace("A3: ", "A3\n")
    text = text.replace("A4: ", "A4\n")
    text = text.replace("A5: ", "A5\n")
    text = text.replace("A6: ", "A6\n")
    text = text.replace("A7: ", "A7\n")
    text = text.replace("A8: ", "A8\n")

    return text


def parse_mean_std(value):
    """
    Converts strings like '0.9070 ± 0.0090' into 0.9070.
    """
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.number)):
        return float(value)

    value = str(value).strip()

    if "±" in value:
        value = value.split("±")[0].strip()

    value = value.replace("+", "")

    try:
        return float(value)
    except Exception:
        return np.nan


def find_col(df, candidates):
    """
    Finds a column using exact or partial matching.
    """
    lower_map = {c.lower(): c for c in df.columns}

    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    for c in df.columns:
        c_low = c.lower()
        for cand in candidates:
            if cand.lower() in c_low:
                return c

    return None


def normalize_ablation_dataframe(df):
    """
    Normalizes either ablation_summary.csv or ablation_manuscript_table.csv
    into a standard plotting dataframe.
    """
    out = pd.DataFrame()

    setting_col = find_col(df, ["Feature setting", "setting", "description", "feature_setting"])
    if setting_col is None:
        raise ValueError("Could not find feature setting column.")

    # Prefer setting + description if both exist
    if "setting" in df.columns and "description" in df.columns:
        out["setting"] = df["setting"].astype(str) + ": " + df["description"].astype(str)
    else:
        out["setting"] = df[setting_col].astype(str)

    out["setting_short"] = out["setting"].apply(clean_setting_label)

    # Feature group boolean columns
    group_map = {
        "lexical_structural": ["lexical_structural", "Lexical/structural", "lexical"],
        "brand_adversarial": ["brand_adversarial", "Brand/adversarial", "adversarial", "brand"],
        "turkish_linguistic": ["turkish_linguistic", "Turkish linguistic", "turkish"],
        "graph_infrastructure": ["graph_infrastructure", "Graph infrastructure", "graph"],
    }

    for std_col, candidates in group_map.items():
        col = find_col(df, candidates)
        if col is not None:
            out[std_col] = df[col].apply(
                lambda x: True if str(x).strip().lower() in ["true", "1", "✓", "yes"] else False
            )
        else:
            # Infer from setting text
            if std_col == "lexical_structural":
                out[std_col] = out["setting"].str.lower().str.contains("lexical")
            elif std_col == "brand_adversarial":
                out[std_col] = out["setting"].str.lower().str.contains("brand|adversarial")
            elif std_col == "turkish_linguistic":
                out[std_col] = out["setting"].str.lower().str.contains("turkish")
            elif std_col == "graph_infrastructure":
                out[std_col] = out["setting"].str.lower().str.contains("graph")

    metric_candidates = {
        "binary_auc": ["binary_auc_mean", "Binary AUC", "binary_auc", "auc"],
        "binary_f1": ["binary_f1_mean", "Binary F1", "binary_f1"],
        "mcc": ["mcc_mean", "MCC", "mcc"],
        "macro_f1": ["five_class_macro_f1_mean", "Five-class macro-F1", "macro_f1", "five_class_macro_f1"],
        "weighted_f1": ["five_class_weighted_f1_mean", "Five-class weighted-F1", "weighted_f1", "five_class_weighted_f1"],
        "delta_macro_f1": ["delta_macro_f1_vs_baseline", "Δ macro-F1 vs A1", "delta_macro_f1"],
        "inference_ms": ["inference_time_per_url_ms_mean", "Inference ms/URL", "inference_ms"],
    }

    for std_col, candidates in metric_candidates.items():
        col = find_col(df, candidates)
        if col is not None:
            out[std_col] = df[col].apply(parse_mean_std)
        else:
            out[std_col] = np.nan

    # If delta missing, compute relative to first row
    if out["delta_macro_f1"].isna().all() and not out["macro_f1"].isna().all():
        baseline = out["macro_f1"].iloc[0]
        out["delta_macro_f1"] = out["macro_f1"] - baseline

    return out


def load_ablation_results():
    """
    Loads ablation outputs from the new script or fallback notebook output.
    """
    if os.path.exists(PRIMARY_SUMMARY):
        print(f"Loading: {PRIMARY_SUMMARY}")
        raw = pd.read_csv(PRIMARY_SUMMARY)
        return normalize_ablation_dataframe(raw)

    if os.path.exists(PRIMARY_TABLE):
        print(f"Loading: {PRIMARY_TABLE}")
        raw = pd.read_csv(PRIMARY_TABLE)
        return normalize_ablation_dataframe(raw)

    if os.path.exists(NOTEBOOK_STEP5):
        print(f"Loading fallback notebook file: {NOTEBOOK_STEP5}")
        raw = pd.read_csv(NOTEBOOK_STEP5)
        return normalize_ablation_dataframe(raw)

    raise FileNotFoundError(
        "No ablation result file found. Expected one of:\n"
        f"1. {PRIMARY_SUMMARY}\n"
        f"2. {PRIMARY_TABLE}\n"
        f"3. {NOTEBOOK_STEP5}\n"
    )


# ============================================================
# 3. LOAD DATA
# ============================================================

plot_df = load_ablation_results()

# Remove rows with no macro-F1
plot_df = plot_df.dropna(subset=["macro_f1"]).reset_index(drop=True)

print("\nLoaded ablation rows:")
print(plot_df[["setting_short", "binary_auc", "mcc", "macro_f1", "delta_macro_f1", "inference_ms"]])


# ============================================================
# 4. PLOT 1 — FIVE-CLASS MACRO-F1 BY FEATURE SETTING
# ============================================================

plt.figure(figsize=(11, 5.5))
x = np.arange(len(plot_df))

plt.bar(x, plot_df["macro_f1"])
plt.xticks(x, plot_df["setting_short"], rotation=35, ha="right")
plt.ylabel("Five-class macro-F1")
plt.title("Feature-group ablation: five-class macro-F1")
plt.grid(axis="y", alpha=0.25)

for i, v in enumerate(plot_df["macro_f1"]):
    plt.text(i, v + 0.002, f"{v:.3f}", ha="center", va="bottom", fontsize=9)

save_fig("fig_ablation_macro_f1")


# ============================================================
# 5. PLOT 2 — DELTA MACRO-F1 VS BASELINE
# ============================================================

plt.figure(figsize=(11, 5.5))
x = np.arange(len(plot_df))

plt.axhline(0, linewidth=1)
plt.bar(x, plot_df["delta_macro_f1"])
plt.xticks(x, plot_df["setting_short"], rotation=35, ha="right")
plt.ylabel("Δ five-class macro-F1 vs lexical baseline")
plt.title("Incremental contribution relative to lexical/structural baseline")
plt.grid(axis="y", alpha=0.25)

for i, v in enumerate(plot_df["delta_macro_f1"]):
    plt.text(
        i,
        v + (0.001 if v >= 0 else -0.001),
        f"{v:+.4f}",
        ha="center",
        va="bottom" if v >= 0 else "top",
        fontsize=9,
    )

save_fig("fig_ablation_delta_macro_f1")


# ============================================================
# 6. PLOT 3 — MULTI-METRIC COMPARISON
# ============================================================

metrics = ["binary_auc", "mcc", "macro_f1", "weighted_f1"]
metric_labels = ["Binary AUC", "MCC", "Macro-F1", "Weighted-F1"]

available_metrics = [m for m in metrics if not plot_df[m].isna().all()]
available_labels = [metric_labels[metrics.index(m)] for m in available_metrics]

plt.figure(figsize=(12, 6))
x = np.arange(len(plot_df))
width = 0.8 / max(1, len(available_metrics))

for j, metric in enumerate(available_metrics):
    offset = (j - (len(available_metrics) - 1) / 2) * width
    plt.bar(x + offset, plot_df[metric], width=width, label=available_labels[j])

plt.xticks(x, plot_df["setting_short"], rotation=35, ha="right")
plt.ylabel("Score")
plt.title("Ablation comparison across evaluation metrics")
plt.ylim(0, 1.05)
plt.grid(axis="y", alpha=0.25)
plt.legend(ncol=2)

save_fig("fig_ablation_metrics_comparison")


# ============================================================
# 7. PLOT 4 — EFFICIENCY TRADE-OFF
# ============================================================

if not plot_df["inference_ms"].isna().all():
    plt.figure(figsize=(8, 5.5))

    plt.scatter(plot_df["inference_ms"], plot_df["macro_f1"], s=80)

    for _, row in plot_df.iterrows():
        label = str(row["setting_short"]).split("\n")[0]
        plt.annotate(
            label,
            (row["inference_ms"], row["macro_f1"]),
            textcoords="offset points",
            xytext=(6, 6),
            fontsize=8,
        )

    plt.xlabel("Inference time per URL (ms)")
    plt.ylabel("Five-class macro-F1")
    plt.title("Accuracy–efficiency trade-off across feature settings")
    plt.grid(alpha=0.25)

    save_fig("fig_ablation_efficiency_tradeoff")
else:
    print("Skipping efficiency trade-off plot because inference_ms is unavailable.")


# ============================================================
# 8. PLOT 5 — FEATURE-GROUP INCLUSION MATRIX
# ============================================================

group_cols = [
    "lexical_structural",
    "brand_adversarial",
    "turkish_linguistic",
    "graph_infrastructure",
]

group_labels = [
    "Lexical/\nstructural",
    "Brand/\nadversarial",
    "Turkish\nlinguistic",
    "Graph\ninfrastructure",
]

matrix = plot_df[group_cols].astype(int).values

plt.figure(figsize=(8.5, 5.5))
plt.imshow(matrix, aspect="auto", interpolation="nearest")

plt.yticks(np.arange(len(plot_df)), plot_df["setting_short"])
plt.xticks(np.arange(len(group_labels)), group_labels)
plt.title("Feature groups included in each ablation setting")

for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        text = "✓" if matrix[i, j] == 1 else "—"
        plt.text(j, i, text, ha="center", va="center", fontsize=13)

plt.xlabel("Feature group")
plt.ylabel("Ablation setting")

save_fig("fig_ablation_feature_group_matrix")


# ============================================================
# 9. PLOT 6 — CONTRIBUTION SUMMARY
# ============================================================

"""
This plot compares selected single-addition settings against the lexical baseline:
A2: + brand/adversarial
A3: + Turkish linguistic
A4: + graph infrastructure

If exact A2/A3/A4 labels are not present, the script tries to infer them.
"""

baseline_macro = plot_df["macro_f1"].iloc[0]

def find_setting_contains(required_terms, forbidden_terms=None):
    if forbidden_terms is None:
        forbidden_terms = []

    for idx, row in plot_df.iterrows():
        text = row["setting"].lower()
        if all(term.lower() in text for term in required_terms) and not any(term.lower() in text for term in forbidden_terms):
            return row

    return None


brand_row = find_setting_contains(["brand"], forbidden_terms=["turkish", "graph"])
turkish_row = find_setting_contains(["turkish"], forbidden_terms=["brand", "graph"])
graph_row = find_setting_contains(["graph"], forbidden_terms=["brand", "turkish", "full"])

contrib_rows = []

if brand_row is not None:
    contrib_rows.append(("Brand/adversarial", brand_row["macro_f1"] - baseline_macro))

if turkish_row is not None:
    contrib_rows.append(("Turkish linguistic", turkish_row["macro_f1"] - baseline_macro))

if graph_row is not None:
    contrib_rows.append(("Graph infrastructure", graph_row["macro_f1"] - baseline_macro))

if contrib_rows:
    contrib_df = pd.DataFrame(contrib_rows, columns=["Feature group added", "Delta macro-F1"])

    plt.figure(figsize=(8, 5))
    x = np.arange(len(contrib_df))

    plt.axhline(0, linewidth=1)
    plt.bar(x, contrib_df["Delta macro-F1"])
    plt.xticks(x, contrib_df["Feature group added"], rotation=20, ha="right")
    plt.ylabel("Δ five-class macro-F1 vs lexical baseline")
    plt.title("Single feature-group contribution over lexical baseline")
    plt.grid(axis="y", alpha=0.25)

    for i, v in enumerate(contrib_df["Delta macro-F1"]):
        plt.text(
            i,
            v + (0.001 if v >= 0 else -0.001),
            f"{v:+.4f}",
            ha="center",
            va="bottom" if v >= 0 else "top",
            fontsize=10,
        )

    save_fig("fig_ablation_contribution_summary")
else:
    print("Skipping contribution summary plot because A2/A3/A4-style rows were not detected.")


# ============================================================
# 10. SAVE PLOTTING DATA
# ============================================================

plot_data_path = os.path.join(OUT_DIR, "ablation_plot_data.csv")
plot_df.to_csv(plot_data_path, index=False)
print(f"\nSaved plotting data: {plot_data_path}")

print("\nDone. All available ablation plots were generated.")

Loading: /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/ablation_outputs/ablation_summary.csv

Loaded ablation rows:
                                      setting_short  binary_auc       mcc  \
0                                       A1\nLexical    0.995622  0.900870   
1                               A2\nLexical + Brand    0.995955  0.904751   
2                             A3\nLexical + Turkish    0.996721  0.915212   
3                               A4\nLexical + Graph    0.998290  0.948358   
4          A5\nLexical + Brand + Turkish linguistic    0.996925  0.916783   
5  A6\nLexical + Brand + Turkish linguistic + graph    0.998693  0.953096   
6                                 A7\nFull retained    0.998693  0.953090   
7     A8\nFull retained, inductive graph projection    0.998693  0.953090   

   macro_f1  delta_macro_f1  inference_ms  
0  0.851296        0.000000      0.086739  
1  0.868620        0.017323      0.082515  
2  0.875574        0.024277      0.08711